In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLNetTokenizer, XLNetModel
print("Libraries and Utils Imported")

Libraries and Utils Imported


## Reading the Data

In [2]:
df = pd.read_csv(r"cleaned_political_bias_data (1).csv")
df.sample(3)

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
7789,2598,ben shapiro stirs controversy by guest writing...,conservative media figure ben shapiro drew con...,center,center,controversy after ben shapiro guest writes a n...,media industry,conservative media figure ben shapiro drew con...
3085,1028,under pressure trump calls new nazis and kkt c...,us president donald trump denounced new nazis ...,center,center,trump condemns hate groups,polarization,us president donald trump denounced new nazis ...
1440,480,come mum on email suggesting move to freeze ou...,a newly declassified email that former nationa...,lean right,conservative,susan rice email adds layer to flynn controversy,national security,a newly declassified email that former nationa...


In [4]:
print("Total Samples:",len(df))
print(df['Stance'].value_counts())
print(df['Label'].value_counts())

Total Samples: 9190
Stance
center        3054
lean left     2237
lean right    2088
right          974
left           829
mixed            6
not rated        2
Name: count, dtype: int64
Label
liberal         3066
conservative    3062
center          3062
Name: count, dtype: int64


Hence Data is Evenly Distributed, No Imbalance of Data

## Setting up Stance to Ideology

This is setup since XLNet is an Encoder and is trained to extract contextual embeddings best from sentences.

In [5]:
## Setting up Stance to Ideology as a dictionary
stance_to_ideology = {
    "left" : "far left leaning ideology",
    'lean left' : 'slightly left leaning ideology',
    'center'    : 'centrist ideology with balanced reporting',
    'lean right': 'slightly right leaning ideology',
    'right'     : 'far right leaning ideology',
    'mixed'     : 'mixed or undefined ideological leaning',
    'not rated' : 'ideology not rated or unknown',
}

## Setting Up the XLNet Configuration

In [14]:
from transformers import XLNetConfig, XLNetModel
from torch.utils.data import Dataset, DataLoader

# Initializing XLNet Tokenizer and Model
XLNet_model = XLNetModel.from_pretrained('xlnet-base-cased')
Tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# Max Sequence Length (if needed more can be adjusted)
max_len = 64

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        stance_text = str(row['Stance_Ideology']) # Stringifying the Stance row
        issue_text = str(row['issue']) # Stringifying the Issue row
        topic_text = str(row['topic']) # Stringifying the Topic row

        encoding_stance = self.tokenizer(
            stance_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        encoding_issue = self.tokenizer(
            issue_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        encoding_topic = self.tokenizer(
            topic_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        return {
            'input_ids_stance': encoding_stance['input_ids'].flatten(),
            'attention_mask_stance': encoding_stance['attention_mask'].flatten(),
            'input_ids_issue': encoding_issue['input_ids'].flatten(),
            'attention_mask_issue': encoding_issue['attention_mask'].flatten(),
            'input_ids_topic': encoding_topic['input_ids'].flatten(),
            'attention_mask_topic': encoding_topic['attention_mask'].flatten()
        }

# Returning a Dictionary of Attention Mask Matrices & Embedded Matrices from XLNet Tokenizer

print("CustomDataset class defined.")

CustomDataset class defined.


### XLNet-based Model with Linear Projection Block
Here is the implementation of XLNet model which processes the three text inputs separately and then projects the concatenated embeddings to 256 dimensions using `LinearProjectionBlock`.

### Linear Projection Block

In [16]:
class LinearProjectionBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dim:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(p=dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.BatchNorm1d(output_dim))
        # layers.append(nn.ReLU(inplace = True)) # Last-Layer does not use RELU as we need feature vector
        self.projection = nn.Sequential(*layers)

    def forward(self, x):
        return self.projection(x)

print("Linear Projection Block defined.")

Linear Projection Block defined.


### The Complete Triple Stance/Topic/Issue Encoder

In [17]:
class XLNetEncoder(nn.Module):

  def __init__(self,model_name,hidden_dim,output_dim,dropout_rate=0.5):
    super().__init__()
    self.xlnet = XLNetModel.from_pretrained(model_name)
    hidden_size = self.xlnet.config.hidden_size #768 for xlnet-base-cased
    self.projection = LinearProjectionBlock(
        input_dim=hidden_size,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        dropout_rate=dropout_rate,
    )

  def forward(self,input_ids,attention_mask):
    outputs = self.xlnet(input_ids=input_ids,attention_mask=attention_mask)
    hidden = outputs.last_hidden_state
    pooled = hidden[:, -1, :]
    return self.projection(pooled)

# Constructor takes: model_name, hidden_dim (intermediate sizes inside LinearProjectionBlock,
# e.g. [512]), output_dim (256), and dropout_rate.
# input_dim (768) is read automatically from XLNet's config — not passed in.
# Then for Forward Propagation, We use input ids and attention masks which we get from Custom Dataset and
# We Return a (Batchsize x 256) dimension Vector

print("XLNetEncoder defined.")

XLNetEncoder defined.


## Now the Combined Stance-Topic-Issue XLNet Encoders

In [24]:
class XLNetTripleEncoder(nn.Module):

  def __init__(self,model_name,hidden_dim,output_dim=256,dropout_rate=0.5):
    super().__init__()

    encoder_kwargs = {
        "model_name":model_name,
        "hidden_dim":hidden_dim,
        "output_dim":output_dim,
        "dropout_rate":dropout_rate
    }

    print("Stance XLNet Encoder")
    self.stance_encoder = XLNetEncoder(**encoder_kwargs)
    print("Issue XLNet Encoder")
    self.issue_encoder = XLNetEncoder(**encoder_kwargs)
    print("Topic XLNet Encoder")
    self.topic_encoder = XLNetEncoder(**encoder_kwargs)

    print("All Encoders Initialized")
    print("Output dimensions=",output_dim)
    print("Total Parameters:",sum(p.numel() for p in self.parameters()))


  def forward(self,batch):
    stance_vec = self.stance_encoder(
        input_ids=batch['input_ids_stance'],
        attention_mask=batch['attention_mask_stance']
    )
    issue_vec = self.issue_encoder(
        input_ids=batch['input_ids_issue'],
        attention_mask=batch['attention_mask_issue']
    )
    topic_vec = self.topic_encoder(
        input_ids=batch['input_ids_topic'],
        attention_mask=batch['attention_mask_topic']
    )
    return {
        'stance_vec':stance_vec,  # (B, 256)
        'issue_vec':issue_vec,  # (B, 256)
        'topic_vec':topic_vec,  # (B, 256)
    }


print("TripleXLNetEncoder Class Instantiated!")
# XLNetTripleEncoder wraps three independent XLNetEncoders — one each for Stance, Issue and Topic.
# Each encoder runs its own XLNet backbone followed by the LinearProjectionBlock.
# Forward pass takes a batch from CustomDataset and passes each attribute
# through its corresponding encoder.
# Returns a dictionary of three independent 256-dimensional context vectors:
# stance_vec (B, 256), issue_vec (B, 256), topic_vec (B, 256)

TripleXLNetEncoder Class Instantiated!


### Dataset & DataLoader

In [27]:
# Create Dataset and DataLoader
dataset    = CustomDataset(df, Tokenizer, max_len)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Instantiate the model
model = XLNetTripleEncoder(
    model_name   = 'xlnet-base-cased',  # string — each XLNetEncoder loads its own backbone
    hidden_dim   = [512],               # LinearProjectionBlock: 768 → 512 → 256
    output_dim   = 256,
    dropout_rate = 0.5,
)

print("Model instantiated:")
print(model)

Stance XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Issue XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Topic XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All Encoders Initialized
Output dimensions= 256
Total Parameters: 351734784
Model instantiated:
XLNetTripleEncoder(
  (stance_encoder): XLNetEncoder(
    (xlnet): XLNetModel(
      (word_embedding): Embedding(32000, 768)
      (layer): ModuleList(
        (0-11): 12 x XLNetLayer(
          (rel_attn): XLNetRelativeAttention(
            (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (ff): XLNetFeedForward(
            (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (layer_1): Linear(in_features=768, out_features=3072, bias=True)
            (layer_2): Linear(in_features=3072, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (activation_function): GELUActivation()
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (projection): Lin

## My Device Details

In [29]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(DEVICE)
print(f'Running on: {DEVICE}')

Running on: cpu


# Inference — Extract 256-dim Context Vectors for Full Dataset

In [ ]:
'''
model.eval()  # Disable dropout & set BatchNorm to eval mode

# Storage for all three streams across all batches
all_stance_vecs = []
all_issue_vecs  = []
all_topic_vecs  = []

with torch.no_grad():  # No gradients needed during inference — saves memory
    for i, batch in enumerate(dataloader):

        # Move batch tensors to device
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # Forward pass through XLNetTripleEncoder
        outputs = model(batch)

        # Move results back to CPU immediately to free GPU memory
        all_stance_vecs.append(outputs['stance_vec'].cpu())
        all_issue_vecs.append(outputs['issue_vec'].cpu())
        all_topic_vecs.append(outputs['topic_vec'].cpu())

        if (i + 1) % 10 == 0:
            print(f'  Processed {(i + 1) * dataloader.batch_size} / {len(dataset)} samples...')

# Concatenate all batches → final matrices for the full dataset
stance_matrix = torch.cat(all_stance_vecs, dim=0)  # (N, 256)
issue_matrix  = torch.cat(all_issue_vecs,  dim=0)  # (N, 256)
topic_matrix  = torch.cat(all_topic_vecs,  dim=0)  # (N, 256)

print('\nInference Complete!')
print(f'Stance Matrix : {stance_matrix.shape}')  # (total_samples, 256)
print(f'Issue Matrix  : {issue_matrix.shape}')   # (total_samples, 256)
print(f'Topic Matrix  : {topic_matrix.shape}')   # (total_samples, 256)
'''

### Verify — Sample Vector Statistics

In [ ]:
# Quick sanity check on the first sample of each stream
'''
print(f"{'Stream':<14} {'Shape':<18} {'Norm':>8} {'Mean':>8} {'Std':>8}")
print('-' * 60)
for name, matrix in [('stance_matrix', stance_matrix), ('issue_matrix', issue_matrix), ('topic_matrix', topic_matrix)]:
    v0 = matrix[0]
    print(f"{name:<14} {str(tuple(matrix.shape)):<18} "
          f"{v0.norm().item():8.4f} "
          f"{v0.mean().item():8.4f} "
          f"{v0.std().item():8.4f}")
'''